# Evaluation Notebook

This file performs evaluation of all algorithms/variants, stores results in a Zarr file, and generates plots.

## Configuration

In [1]:
# Set the output file automatically.
from datetime import datetime
datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = "./results/patrolling_comms_draft2_test20_" + datetime_str + ".zarr"

# output_file = "./results/patrolling_comms_draft2_test20_pareto.zarr"
# output_file = "/data/anthony/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_draft2_test20_20260523_154041.zarr"
output_file = '/home/anthony/data/dev/qmas_task_allocation/scripts/final/results/patrolling_comms_draft2_test20_20260523_162527.zarr'

## Setup

In [2]:
%reload_ext autoreload
%autoreload 2
from onpolicy.scripts.eval.eval_pettingzoo import main
import os
import time
from multiprocessing import Process
from args import *

MAX_PROCESSES = 600
GPUS = [0, 1, 2, 3, 4, 5, 6, 7]

def get_gpu_least_used():
    gpu_memory = os.popen("nvidia-smi --query-gpu=memory.used --format=csv").read()
    gpu_memory = gpu_memory.split("\n")[1:-1]
    gpu_memory = [int(x.split()[0]) for x in gpu_memory]
    gpu_memory = [gpu_memory[i] if i in GPUS else float('inf') for i in range(len(gpu_memory))]
    least_used_gpu = gpu_memory.index(min(gpu_memory))
    return least_used_gpu

# Set default arguments for evaluation. Feel free to change these!
default_args = {}
default_args["eval_output_file"] = wrap_arg_val(output_file)
default_args["cuda"] = wrap_arg_val(True)
default_args["cuda_idx"] = wrap_arg_val(1)
default_args["cuda_idx_predictor"] = default_args["cuda_idx"]
default_args["eval_episodes"] = wrap_arg_val(10)
default_args["episode_length"] = wrap_arg_val(150)
default_args["seed"] = wrap_arg_val(2)

default_args["observation_mask"] = wrap_arg_val(True)

# default_args["diffusion_autoregression_steps"] = wrap_arg_val(1)

# default_args["graph_name"] = wrap_arg_val("cumberland")
# default_args["graph_file"] = wrap_arg_val("../../patrolling_zoo/graphs/cumberland.graph")
default_args["graph_random"] = wrap_arg_val(True)

default_args["observation_radius_random_min"] = wrap_arg_val(0.0)
default_args["observation_radius_random_max"] = wrap_arg_val(0.0)

# Evaluation

In [3]:
import numpy as np
# obs_radii = [0.0, 50.0, 100.0, 150.0, 200.0, 250.0, 300.0]
obs_radii = np.arange(20.0, 161.0, 20.0).tolist()
# obs_radii = np.arange(5.0, 51.0, 10.0).tolist()
comms_probabilities = np.arange(0.0, 0.51, 0.05).tolist()
# comms_probabilities = np.arange(0.0, 0.51, 0.1).tolist()

# Check if the output file already exists. If so, skip this cell.
if os.path.exists(output_file):
    print("Output file already exists. Skipping evaluation.")

else:
    process_args = []

    def add_process(args):
        process_args.append(args)
    
    def start_process(args):
        p = Process(target=main, args=(args,))
        p.start()
        return p

    for obs_radius in obs_radii:
        for comms_p in comms_probabilities:
            # No observation masking.
            args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-draft1/wandb/run-20260515_192751-jdsn65y4/files", default_args)
            args["eval_series"] = "MAGEC Ground Truth (Fixed Rate)"
            args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            args["communication_probability"] = wrap_arg_val(comms_p)
            args["observation_mask"] = wrap_arg_val(False)
            add_process(args)

            args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-draft1/wandb/run-20260521_023023-fglklioe/files", default_args)
            args["eval_series"] = "MAGEC (Fixed Rate)"
            args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            args["communication_probability"] = wrap_arg_val(comms_p)
            add_process(args)

            args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-pred-draft1/wandb/run-20260521_023023-xcdnjkc1/files", default_args)
            args["eval_series"] = "Predictions (Fixed Rate)"
            args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
            args["observation_radius"] = wrap_arg_val(obs_radius)
            args["communication_probability"] = wrap_arg_val(comms_p)
            add_process(args)

            # args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-pred-uqstd-draft1/wandb/run-20260521_023023-fv3k32hm/files", default_args)
            # args["eval_series"] = "Prediction + UQ"
            # args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
            # args["observation_radius"] = wrap_arg_val(obs_radius)
            # args["communication_probability"] = wrap_arg_val(comms_p)
            # add_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-eozh4t4o/files", default_args)
        args["eval_series"] = "Predictions+UQ (Threshold, T=1.0)"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        add_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-qd2nyd6h/files", default_args)
        args["eval_series"] = "Predictions+UQ (Threshold, T=0.5)"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_p)
        add_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-muk12tx1/files", default_args)
        args["eval_series"] = "Predictions+UQ (Threshold, T=0.1)"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}_comms_{comms_p:.3f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        args["communication_probability"] = wrap_arg_val(comms_p)
        add_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-draft1/wandb/run-20260521_023023-9sgybv9e/files", default_args)
        args["eval_series"] = f"Observations (Boolean)"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        add_process(args)

        args = load_args("/data/group/mas/qmas/results/patrolling_zoo_v0/mappo/patrolling-randomOR-comms-pred-uqstd-draft1/wandb/run-20260521_023023-7pgsd7xf/files", default_args)
        args["eval_series"] = f"Predictions+UQ (Boolean)"
        args["experiment_name"] = wrap_arg_val(f"obs_radius_{obs_radius:.1f}")
        args["observation_radius"] = wrap_arg_val(obs_radius)
        add_process(args)

    print(f"Prepared {len(process_args)} processes to run.")

    # JOIN ALL PROCESSES
    # Up to MAX_PROCESSES processes can run simultaneously.
    running_processes = []
    time_start = time.time()
    failed_processes = []
    
    while len(process_args) > 0 or len(running_processes) > 0:
        # Start new processes if we have capacity.
        while len(running_processes) < MAX_PROCESSES and len(process_args) > 0:
            args = process_args.pop(0)

            # Get least used GPU and set it in args.
            least_used_gpu = get_gpu_least_used()
            args["cuda_idx"] = wrap_arg_val(least_used_gpu)
            args["cuda_idx_predictor"] = wrap_arg_val(least_used_gpu)

            p = start_process(convert_args(args))
            running_processes.append(p)
        
        # Check for finished processes.
        for p in running_processes:
            if not p.is_alive():
                p.join()
                print(f"Process {p.pid} finished with exit code {p.exitcode}. Running: {len(running_processes)-1}, queued: {len(process_args)}, failed: {len(failed_processes)}")
                if p.exitcode != 0:
                    failed_processes.append(p)
        running_processes = [p for p in running_processes if p.is_alive()]
    
    time_end = time.time()
    print(f"Total evaluation time: {time_end - time_start:.2f} seconds")
    
    if failed_processes:
        raise RuntimeError(f"{len(failed_processes)} subprocess(es) failed with non-zero exit codes")
    
    print("All processes finished successfully!")

Output file already exists. Skipping evaluation.


# Analysis

In [4]:
import zarr
output = zarr.open(output_file, mode='r')
print(output.tree())
print(f"Output file: {os.path.abspath(output_file)}")
print(f"To load results, set in first cell: output_file = '{os.path.abspath(output_file)}'")

/
 ├── MAGEC (Fixed Rate)
 │   ├── obs_radius_100.0_comms_0.000
 │   │   ├── agent_count (10, 150) float32
 │   │   ├── avg_idleness (10, 150) float32
 │   │   ├── communication
 │   │   │   └── requests_made (10, 150) float32
 │   │   ├── deltaSteps (10, 150, 4, 1) float32
 │   │   ├── node_visits
 │   │   │   ├── node_0 (10, 150) float32
 │   │   │   ├── node_1 (10, 150) float32
 │   │   │   ├── node_10 (10, 150) float32
 │   │   │   ├── node_11 (10, 150) float32
 │   │   │   ├── node_12 (10, 150) float32
 │   │   │   ├── node_13 (10, 150) float32
 │   │   │   ├── node_14 (10, 150) float32
 │   │   │   ├── node_15 (10, 150) float32
 │   │   │   ├── node_16 (10, 150) float32
 │   │   │   ├── node_17 (10, 150) float32
 │   │   │   ├── node_18 (10, 150) float32
 │   │   │   ├── node_19 (10, 150) float32
 │   │   │   ├── node_2 (10, 150) float32
 │   │   │   ├── node_20 (10, 150) float32
 │   │   │   ├── node_21 (10, 150) float32
 │   │   │   ├── node_22 (10, 150) float32
 │   │   │   ├─

## Idleness vs. Communication

In [21]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from plots import *

# SERIES_TO_PLOT = {"Learned Comms. + Prediction + UQ", "MAGEC"}
SERIES_TO_PLOT = None # Plot all

series_colors = {s: c for s, c in zip(output, px.colors.qualitative.Plotly)}


def get_requests(data):
    """Total communication requests per episode (mean over episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    return float(np.mean(r.reshape(r.shape[0], -1).sum(axis=1)))


def get_requests_std(data):
    """Total communication requests per episode (SEM over episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    per_episode = r.reshape(r.shape[0], -1).sum(axis=1)
    n = len(per_episode)
    return float(np.std(per_episode) / np.sqrt(n))


def get_idleness_mean(data):
    """Time-averaged avg. idleness, mean over episodes."""
    return float(np.mean(data['avg_idleness']))


def get_idleness_std(data):
    """Time-averaged avg. idleness, SEM over episodes."""
    n = len(data['avg_idleness'])
    return float(np.std(data['avg_idleness']) / np.sqrt(n))


for obs_radius in obs_radii:
    # Arbitrarily choose radius 40 for plotting.
    if obs_radius != 40.0:
        continue

    fig = go.Figure()
    r_annotations = []

    for series in output:
        if SERIES_TO_PLOT is not None and series not in SERIES_TO_PLOT:
            continue

        if "Ground Truth" in series:
            continue

        color = series_colors.get(series, '#888888')
        all_exps = list(output[series].keys())

        # Swept experiments: encoded as obs_radius_X_comms_Y
        swept_exps = sorted(
            [e for e in all_exps if f"obs_radius_{obs_radius:.1f}_comms_" in e and "Fixed Rate" in series],
            key=lambda e: float(e.split("_comms_")[1])
        )
        # Fixed (learned comms) experiments: encoded as obs_radius_X only
        fixed_exps = [e for e in all_exps if f"obs_radius_{obs_radius:.1f}" in e and not "Fixed Rate" in series]

        if swept_exps:
            comms_ps = [float(e.split("_comms_")[1]) for e in swept_exps]
            xs = [get_requests(output[series][e]) for e in swept_exps]
            ys = [get_idleness_mean(output[series][e]) for e in swept_exps]
            xs_std = [get_requests_std(output[series][e]) for e in swept_exps]
            ys_std = [get_idleness_std(output[series][e]) for e in swept_exps]

            fig.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=8),
                legendgroup=series,
                # error_x and error_y removed — replaced by shaded band below
                hovertemplate=[
                    f"{series}<br>comms_p={cp:.3f}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>"
                    for cp in comms_ps
                ],
            ))

            # SEM shaded band (y-direction)
            y_upper = [y + s for y, s in zip(ys, ys_std)]
            y_lower = [y - s for y, s in zip(ys, ys_std)]
            fig.add_trace(go.Scatter(
                x=xs + xs[::-1],
                y=y_upper + y_lower[::-1],
                fill='toself',
                fillcolor=color,
                line=dict(color='rgba(255,255,255,0)'),
                opacity=0.15,
                legendgroup=series,
                showlegend=False,
                hoverinfo='skip',
            ))

            # OLS trendline
            if len(xs) >= 2:
                x_trend, y_trend, r2 = fit_ols_trendline(xs, ys)
                fig.add_trace(go.Scatter(
                    x=x_trend,
                    y=y_trend,
                    mode='lines',
                    name=f"{series} (fit)",
                    line=dict(color=color, dash='dash', width=1),
                    legendgroup=series,
                    showlegend=False,
                    hoverinfo='skip',
                ))
                r_annotations.append(
                    dict(
                        text=f"<b>{series}</b> R={r2:.3f}",
                        xref='paper', yref='paper',
                        x=0.01,
                        y=0.99 - len(r_annotations) * 0.06,
                        xanchor='left', yanchor='top',
                        showarrow=False,
                        font=dict(color=color, size=12),
                    )
                )

        if fixed_exps:
            xs_f = [get_requests(output[series][e]) for e in fixed_exps]
            ys_f = [get_idleness_mean(output[series][e]) for e in fixed_exps]
            xs_std_f = [get_requests_std(output[series][e]) for e in fixed_exps]
            ys_std_f = [get_idleness_std(output[series][e]) for e in fixed_exps]
            fig.add_trace(go.Scatter(
                x=xs_f,
                y=ys_f,
                mode='markers',
                name=series,
                marker=dict(color=color, size=14, symbol='star'),
                legendgroup=series,
                showlegend=len(swept_exps) == 0,
                error_x=dict(array=xs_std_f, visible=True, color=color, thickness=1.5, width=4),
                error_y=dict(array=ys_std_f, visible=True, color=color, thickness=1.5, width=4),
                hovertemplate=f"{series}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>",
            ))

    fig.update_layout(
        title=f"Time-Averaged Avg. Idleness vs. Communication Requests (obs_radius={obs_radius:.1f})",
        xaxis_title="Total Communication Requests (per episode)",
        yaxis_title="Time-Averaged Avg. Idleness (s)",
        height=600,
        width=800,
        hovermode='closest',
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
        annotations=r_annotations,
    )
    fig.show()


In [22]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from plots import *

# SERIES_TO_PLOT = {"Learned Comms. + Prediction + UQ", "MAGEC"}
SERIES_TO_PLOT = None # Plot all

series_colors = {s: c for s, c in zip(output, px.colors.qualitative.Plotly)}


def get_requests(data):
    """Total communication requests per episode (mean over episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    return float(np.mean(r.reshape(r.shape[0], -1).sum(axis=1)))


def get_requests_std(data):
    """Total communication requests per episode (SEM over episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    per_episode = r.reshape(r.shape[0], -1).sum(axis=1)
    n = len(per_episode)
    return float(np.std(per_episode) / np.sqrt(n))


def get_idleness_mean(data):
    """Time-averaged std. idleness, mean over episodes."""
    return float(np.mean(data['stddev_idleness']))


def get_idleness_std(data):
    """Time-averaged std. idleness, SEM over episodes."""
    n = len(data['stddev_idleness'])
    return float(np.std(data['stddev_idleness']) / np.sqrt(n))


for obs_radius in obs_radii:
    # Arbitrarily choose radius 40 for plotting.
    if obs_radius != 40.0:
        continue

    fig = go.Figure()
    r_annotations = []

    for series in output:
        if SERIES_TO_PLOT is not None and series not in SERIES_TO_PLOT:
            continue

        if "Ground Truth" in series:
            continue

        color = series_colors.get(series, '#888888')
        all_exps = list(output[series].keys())

        # Swept experiments: encoded as obs_radius_X_comms_Y
        swept_exps = sorted(
            [e for e in all_exps if f"obs_radius_{obs_radius:.1f}_comms_" in e and "Fixed Rate" in series],
            key=lambda e: float(e.split("_comms_")[1])
        )
        # Fixed (learned comms) experiments: encoded as obs_radius_X only
        fixed_exps = [e for e in all_exps if f"obs_radius_{obs_radius:.1f}" in e and not "Fixed Rate" in series]

        if swept_exps:
            comms_ps = [float(e.split("_comms_")[1]) for e in swept_exps]
            xs = [get_requests(output[series][e]) for e in swept_exps]
            ys = [get_idleness_mean(output[series][e]) for e in swept_exps]
            xs_std = [get_requests_std(output[series][e]) for e in swept_exps]
            ys_std = [get_idleness_std(output[series][e]) for e in swept_exps]

            fig.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=8),
                legendgroup=series,
                # error_x and error_y removed — replaced by shaded band below
                hovertemplate=[
                    f"{series}<br>comms_p={cp:.3f}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>"
                    for cp in comms_ps
                ],
            ))

            # SEM shaded band (y-direction)
            y_upper = [y + s for y, s in zip(ys, ys_std)]
            y_lower = [y - s for y, s in zip(ys, ys_std)]
            fig.add_trace(go.Scatter(
                x=xs + xs[::-1],
                y=y_upper + y_lower[::-1],
                fill='toself',
                fillcolor=color,
                line=dict(color='rgba(255,255,255,0)'),
                opacity=0.15,
                legendgroup=series,
                showlegend=False,
                hoverinfo='skip',
            ))

            # OLS trendline
            if len(xs) >= 2:
                x_trend, y_trend, r2 = fit_ols_trendline(xs, ys)
                fig.add_trace(go.Scatter(
                    x=x_trend,
                    y=y_trend,
                    mode='lines',
                    name=f"{series} (fit)",
                    line=dict(color=color, dash='dash', width=1),
                    legendgroup=series,
                    showlegend=False,
                    hoverinfo='skip',
                ))
                r_annotations.append(
                    dict(
                        text=f"<b>{series}</b> R={r2:.3f}",
                        xref='paper', yref='paper',
                        x=0.01,
                        y=0.99 - len(r_annotations) * 0.06,
                        xanchor='left', yanchor='top',
                        showarrow=False,
                        font=dict(color=color, size=12),
                    )
                )

        if fixed_exps:
            xs_f = [get_requests(output[series][e]) for e in fixed_exps]
            ys_f = [get_idleness_mean(output[series][e]) for e in fixed_exps]
            xs_std_f = [get_requests_std(output[series][e]) for e in fixed_exps]
            ys_std_f = [get_idleness_std(output[series][e]) for e in fixed_exps]
            fig.add_trace(go.Scatter(
                x=xs_f,
                y=ys_f,
                mode='markers',
                name=series,
                marker=dict(color=color, size=14, symbol='star'),
                legendgroup=series,
                showlegend=len(swept_exps) == 0,
                error_x=dict(array=xs_std_f, visible=True, color=color, thickness=1.5, width=4),
                error_y=dict(array=ys_std_f, visible=True, color=color, thickness=1.5, width=4),
                hovertemplate=f"{series}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>",
            ))

    fig.update_layout(
        title=f"Time-Averaged SD Idleness vs. Communication Requests (obs_radius={obs_radius:.1f})",
        xaxis_title="Total Communication Requests (per episode)",
        yaxis_title="Time-Averaged SD Idleness (s)",
        height=600,
        width=800,
        hovermode='closest',
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
        annotations=r_annotations,
    )
    fig.show()


In [25]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from plots import *

# SERIES_TO_PLOT = {"Learned Comms. + Prediction + UQ", "MAGEC"}
SERIES_TO_PLOT = None # Plot all

series_colors = {s: c for s, c in zip(output, px.colors.qualitative.Plotly)}


def get_requests(data):
    """Total communication requests per episode (mean over episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    return float(np.mean(r.reshape(r.shape[0], -1).sum(axis=1)))


def get_requests_std(data):
    """Total communication requests per episode (SEM over episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    per_episode = r.reshape(r.shape[0], -1).sum(axis=1)
    n = len(per_episode)
    return float(np.std(per_episode) / np.sqrt(n))


def get_idleness_mean(data):
    """Time-averaged worst idleness, mean over episodes."""
    return float(np.mean(data['worst_idleness']))


def get_idleness_std(data):
    """Time-averaged worst idleness, SEM over episodes."""
    n = len(data['worst_idleness'])
    return float(np.std(data['worst_idleness']) / np.sqrt(n))


for obs_radius in obs_radii:
    # Arbitrarily choose radius 40 for plotting.
    if obs_radius != 40.0:
        continue

    fig = go.Figure()
    r_annotations = []

    for series in output:
        if SERIES_TO_PLOT is not None and series not in SERIES_TO_PLOT:
            continue

        if "Ground Truth" in series:
            continue

        color = series_colors.get(series, '#888888')
        all_exps = list(output[series].keys())

        # Swept experiments: encoded as obs_radius_X_comms_Y
        swept_exps = sorted(
            [e for e in all_exps if f"obs_radius_{obs_radius:.1f}_comms_" in e and "Fixed Rate" in series],
            key=lambda e: float(e.split("_comms_")[1])
        )
        # Fixed (learned comms) experiments: encoded as obs_radius_X only
        fixed_exps = [e for e in all_exps if f"obs_radius_{obs_radius:.1f}" in e and not "Fixed Rate" in series]

        if swept_exps:
            comms_ps = [float(e.split("_comms_")[1]) for e in swept_exps]
            xs = [get_requests(output[series][e]) for e in swept_exps]
            ys = [get_idleness_mean(output[series][e]) for e in swept_exps]
            xs_std = [get_requests_std(output[series][e]) for e in swept_exps]
            ys_std = [get_idleness_std(output[series][e]) for e in swept_exps]

            fig.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=8),
                legendgroup=series,
                # error_x and error_y removed — replaced by shaded band below
                hovertemplate=[
                    f"{series}<br>comms_p={cp:.3f}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>"
                    for cp in comms_ps
                ],
            ))

            # SEM shaded band (y-direction)
            y_upper = [y + s for y, s in zip(ys, ys_std)]
            y_lower = [y - s for y, s in zip(ys, ys_std)]
            fig.add_trace(go.Scatter(
                x=xs + xs[::-1],
                y=y_upper + y_lower[::-1],
                fill='toself',
                fillcolor=color,
                line=dict(color='rgba(255,255,255,0)'),
                opacity=0.15,
                legendgroup=series,
                showlegend=False,
                hoverinfo='skip',
            ))

            # OLS trendline
            if len(xs) >= 2:
                x_trend, y_trend, r2 = fit_ols_trendline(xs, ys)
                fig.add_trace(go.Scatter(
                    x=x_trend,
                    y=y_trend,
                    mode='lines',
                    name=f"{series} (fit)",
                    line=dict(color=color, dash='dash', width=1),
                    legendgroup=series,
                    showlegend=False,
                    hoverinfo='skip',
                ))
                r_annotations.append(
                    dict(
                        text=f"<b>{series}</b> R={r2:.3f}",
                        xref='paper', yref='paper',
                        x=0.01,
                        y=0.99 - len(r_annotations) * 0.06,
                        xanchor='left', yanchor='top',
                        showarrow=False,
                        font=dict(color=color, size=12),
                    )
                )

        if fixed_exps:
            xs_f = [get_requests(output[series][e]) for e in fixed_exps]
            ys_f = [get_idleness_mean(output[series][e]) for e in fixed_exps]
            xs_std_f = [get_requests_std(output[series][e]) for e in fixed_exps]
            ys_std_f = [get_idleness_std(output[series][e]) for e in fixed_exps]
            fig.add_trace(go.Scatter(
                x=xs_f,
                y=ys_f,
                mode='markers',
                name=series,
                marker=dict(color=color, size=14, symbol='star'),
                legendgroup=series,
                showlegend=len(swept_exps) == 0,
                error_x=dict(array=xs_std_f, visible=True, color=color, thickness=1.5, width=4),
                error_y=dict(array=ys_std_f, visible=True, color=color, thickness=1.5, width=4),
                hovertemplate=f"{series}<br>requests=%{{x:.1f}}<br>idleness=%{{y:.4f}}<extra></extra>",
            ))

    fig.update_layout(
        title=f"Time-Averaged Worst Idleness vs. Communication Requests (obs_radius={obs_radius:.1f})",
        xaxis_title="Total Communication Requests (per episode)",
        yaxis_title="Time-Averaged Worst Idleness (s)",
        height=600,
        width=800,
        hovermode='closest',
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
        annotations=r_annotations,
    )
    fig.show()


## Idleness vs. Observation Radius

In [6]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from plots import *

# SERIES_TO_PLOT = {"Learned Comms. + Prediction + UQ", "MAGEC"}
SERIES_TO_PLOT = None # Plot all

series_colors = {s: c for s, c in zip(output, px.colors.qualitative.Plotly)}


def get_requests(data):
    """Total communication requests per episode (mean over episodes)."""
    r = np.array(data['communication']['requests_made'][...])
    return float(np.mean(r.reshape(r.shape[0], -1).sum(axis=1)))


def get_idleness_mean(data):
    """Time-averaged avg. idleness, mean over episodes."""
    return float(np.mean(data['avg_idleness']))


def get_idleness_std(data):
    """Time-averaged avg. idleness, SEM over episodes."""
    n = len(data['avg_idleness'])
    return float(np.std(data['avg_idleness']) / np.sqrt(n))

for comms_p in comms_probabilities:
    fig = go.Figure()
    r_annotations = []

    if comms_p > 0.0:
        # We only care about the 0.0 case right now!
        continue

    for series in output:
        if SERIES_TO_PLOT is not None and series not in SERIES_TO_PLOT:
            continue

        name = series.replace("Fixed Rate", f"Comms. Rate {comms_p:.1f}")

        color = series_colors.get(series, '#888888')
        all_exps = list(output[series].keys())

        # Swept experiments: encoded as obs_radius_X_comms_Y, filtered to this comms_p
        swept_exps = sorted(
            [e for e in all_exps if output[series][e].attrs.get("communication_probability", -1.0) == comms_p and "Fixed Rate" in series],
            key=lambda e: output[series][e].attrs.get("observation_radius", 0.0)
        )

        if swept_exps:
            xs = [output[series][e].attrs.get("observation_radius", 0.0) for e in swept_exps]
            ys = [get_idleness_mean(output[series][e]) for e in swept_exps]
            ys_std = [get_idleness_std(output[series][e]) for e in swept_exps]
            y_upper = [y + s for y, s in zip(ys, ys_std)]
            y_lower = [y - s for y, s in zip(ys, ys_std)]

            fig.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='lines+markers',
                name=name,
                line=dict(color=color),
                marker=dict(color=color, size=8),
                legendgroup=series,
                hovertemplate=[
                    f"{series}<br>obs_radius={r:.1f}<br>idleness=%{{y:.4f}}<extra></extra>"
                    for r in xs
                ],
            ))
            # SEM shaded band
            fig.add_trace(go.Scatter(
                x=xs + xs[::-1],
                y=y_upper + y_lower[::-1],
                fill='toself',
                fillcolor=color,
                line=dict(color='rgba(255,255,255,0)'),
                opacity=0.1,
                legendgroup=series,
                showlegend=False,
                hoverinfo='skip',
            ))

            # OLS trendline
            if len(xs) >= 2:
                x_trend, y_trend, r = fit_ols_trendline(xs, ys)
                fig.add_trace(go.Scatter(
                    x=x_trend,
                    y=y_trend,
                    mode='lines',
                    name=f"{name} (fit)",
                    line=dict(color=color, dash='dash', width=1),
                    legendgroup=series,
                    showlegend=False,
                    hoverinfo='skip',
                ))
                r_annotations.append(
                    dict(
                        text=f"<b>{name}</b> R={r:.3f}",
                        xref='paper', yref='paper',
                        x=0.01,
                        y=0.99 - len(r_annotations) * 0.06,
                        xanchor='left', yanchor='top',
                        showarrow=False,
                        font=dict(color=color, size=12),
                    )
                )

    fig.update_layout(
        title=f"Time-Averaged Avg. Idleness vs. Observation Radius",
        xaxis_title="Observation Radius",
        yaxis_title="Time-Averaged Avg. Idleness (s)",
        height=600,
        width=800,
        hovermode='closest',
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
        annotations=r_annotations,
    )
    fig.show()


## Communication vs. Observation Radius

In [7]:
for comms_p in comms_probabilities:
    fig2 = go.Figure()
    r2_annotations2 = []

    for series in output:
        if SERIES_TO_PLOT is not None and series not in SERIES_TO_PLOT:
            continue

        color = series_colors.get(series, '#888888')
        all_exps_s = list(output[series].keys())

        # Swept experiments for this comms_p: obs_radius_X_comms_Y
        swept = sorted(
            [e for e in all_exps_s if f"Fixed Rate" in series and f"_comms_{comms_p:.3f}" in e],
            key=lambda e: output[series][e].attrs.get("observation_radius", float('inf'))
        )
        # Fixed (learned comms) experiments: obs_radius_X only (no comms component)
        fixed = sorted(
            [e for e in all_exps_s if not "Fixed Rate" in series],
            key=lambda e: output[series][e].attrs.get("observation_radius", float('inf'))
        )

        if swept:
            continue
            xs = [float(e.split("obs_radius_")[1].split("_comms_")[0]) for e in swept]
            ys = [get_requests(output[series][e]) for e in swept]

            fig2.add_trace(go.Scatter(
                x=xs,
                y=ys,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=8),
                legendgroup=series,
                hovertemplate=[
                    f"{series}<br>obs_radius={r:.1f}<br>requests=%{{y:.1f}}<extra></extra>"
                    for r in xs
                ],
            ))

            # OLS trendline
            if len(xs) >= 2:
                x_trend2, y_trend2, r2_2 = fit_ols_trendline(xs, ys)
                fig2.add_trace(go.Scatter(
                    x=x_trend2,
                    y=y_trend2,
                    mode='lines',
                    name=f"{series} (fit)",
                    line=dict(color=color, dash='dash', width=1),
                    legendgroup=series,
                    showlegend=False,
                    hoverinfo='skip',
                ))
                r2_annotations2.append(
                    dict(
                        text=f"<b>{series}</b> R²={r2_2:.3f}",
                        xref='paper', yref='paper',
                        x=0.01,
                        y=0.99 - len(r2_annotations2) * 0.06,
                        xanchor='left', yanchor='top',
                        showarrow=False,
                        font=dict(color=color, size=12),
                    )
                )

        if fixed:
            xs_f = [output[series][e].attrs.get("observation_radius", float('inf')) for e in fixed]
            ys_f = [get_requests(output[series][e]) for e in fixed]
            ys_std_f = [get_requests_std(output[series][e]) for e in fixed]
            y_upper_f = [y + s for y, s in zip(ys_f, ys_std_f)]
            y_lower_f = [y - s for y, s in zip(ys_f, ys_std_f)]
            fig2.add_trace(go.Scatter(
                x=xs_f,
                y=ys_f,
                mode='lines+markers',
                name=series,
                line=dict(color=color),
                marker=dict(color=color, size=14, symbol='star'),
                legendgroup=series,
                showlegend=len(swept) == 0,
                hovertemplate=f"{series}<br>obs_radius=%{{x:.1f}}<br>requests=%{{y:.1f}}<extra></extra>",
            ))
            # SEM shaded band
            fig2.add_trace(go.Scatter(
                x=xs_f + xs_f[::-1],
                y=y_upper_f + y_lower_f[::-1],
                fill='toself',
                fillcolor=color,
                line=dict(color='rgba(255,255,255,0)'),
                opacity=0.2,
                legendgroup=series,
                showlegend=False,
                hoverinfo='skip',
            ))

fig2.update_layout(
    title=f"Communication Requests vs. Observation Radius",
    # title=f"Communication Requests vs. Observation Radius (comms_p={comms_p:.3f})",
    xaxis_title="Observation Radius",
    yaxis_title="Total Communication Requests (per episode)",
    height=600,
    width=800,
    hovermode='closest',
    autosize=True,
    margin=dict(l=60, r=20, t=60, b=120),
    legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
    annotations=r2_annotations2,
)
fig2.show()


## Prediction Error vs. Time

In [12]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

PLOT_COMMS_P = 0.0   # 0 = no fixed-rate comms; set to None to include all

series_colors = {s: c for s, c in zip(output, px.colors.qualitative.Plotly)}

for PLOT_OBS_RADIUS in obs_radii:
    fig = go.Figure()

    for series in output:
        color = series_colors.get(series, '#888888')
        all_exps = list(output[series].keys())

        # Collect experiments that contain prediction_error_mean data.
        candidate_exps = []
        for e in all_exps:
            grp = output[series][e]
            if "prediction_error_mean" not in grp:
                continue
            obs_r = grp.attrs.get("observation_radius", None)
            comms_p_val = grp.attrs.get("communication_probability", None)

            if obs_r is not None and abs(obs_r - PLOT_OBS_RADIUS) > 1e-3:
                continue
            if PLOT_COMMS_P is not None and comms_p_val is not None and abs(comms_p_val - PLOT_COMMS_P) > 1e-3:
                continue

            candidate_exps.append(e)

        if not candidate_exps:
            continue

        # Average prediction_error_mean over all selected experiments and all episodes.
        # Shape per experiment: (episodes, episode_length)
        all_timeseries = []
        for e in candidate_exps:
            data = np.array(output[series][e]["prediction_error_mean"][...])  # (episodes, T)
            all_timeseries.append(data)

        # Stack → (n_exps * episodes, T), then take mean and SEM over axis 0.
        stacked = np.concatenate(all_timeseries, axis=0)   # (N, T)
        N = stacked.shape[0]
        mean_ts = stacked.mean(axis=0)                      # (T,)
        sem_ts = stacked.std(axis=0) / np.sqrt(N)           # (T,)
        timesteps = np.arange(len(mean_ts))

        fig.add_trace(go.Scatter(
            x=timesteps,
            y=mean_ts,
            mode='lines',
            name=series,
            line=dict(color=color),
            legendgroup=series,
            hovertemplate=f"{series}<br>t=%{{x}}<br>error=%{{y:.4f}}<extra></extra>",
        ))
        # SEM shaded band
        fig.add_trace(go.Scatter(
            x=np.concatenate([timesteps, timesteps[::-1]]),
            y=np.concatenate([mean_ts + sem_ts, (mean_ts - sem_ts)[::-1]]),
            fill='toself',
            fillcolor=color,
            line=dict(color='rgba(255,255,255,0)'),
            opacity=0.15,
            legendgroup=series,
            showlegend=False,
            hoverinfo='skip',
        ))

    title_suffix = f", obs_radius={PLOT_OBS_RADIUS:.1f}"
    if PLOT_COMMS_P is not None:
        title_suffix += f", comms_p={PLOT_COMMS_P:.2f}"

    fig.update_layout(
        title=f"Mean Prediction Error vs. Time{title_suffix}",
        xaxis_title="Timestep",
        yaxis_title="Mean Absolute Prediction Error",
        height=500,
        width=900,
        hovermode='x unified',
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation='h', yanchor='top', y=-0.2, xanchor='center', x=0.5),
    )
    fig.show()


## Learning Curves (from W&B)

In [29]:
import wandb
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

# ── W&B coordinates ──────────────────────────────────────────────────────────
WANDB_ENTITY  = "ideas-mas"
WANDB_PROJECT = "qmas"

# Map display name → wandb run ID (taken from the wandb run paths used above).
LEARNING_CURVE_RUNS = {
    # "MAGEC Ground Truth":        "jdsn65y4",
    "MAGEC":                     "fglklioe",
    "Predictions":               "xcdnjkc1",
    "Predictions+UQ (T=1.0)":   "eozh4t4o",
    "Predictions+UQ (T=0.5)":   "qd2nyd6h",
    "Predictions+UQ (T=0.1)":   "muk12tx1",
    "Observations (Boolean)":   "9sgybv9e",
    "Predictions+UQ (Boolean)": "7pgsd7xf",
}

# Metrics to plot (wandb key → axis label).
METRICS = {
    "average_episode_rewards": "Average Episode Reward",
    "worst_idleness":          "Worst Idleness",
}

# Rolling-average window applied before plotting (set to 1 to disable).
SMOOTH_WINDOW = 20

# ── Fetch history from W&B ────────────────────────────────────────────────────
api = wandb.Api()

def rolling_mean(arr, window):
    """Centred rolling mean using numpy; handles edges by shrinking the window."""
    out = np.empty_like(arr, dtype=float)
    half = window // 2
    for i in range(len(arr)):
        lo = max(0, i - half)
        hi = min(len(arr), i + half + 1)
        out[i] = np.nanmean(arr[lo:hi])
    return out

run_histories = {}  # name → {"step": ndarray, metric_key: ndarray, ...}
for name, run_id in LEARNING_CURVE_RUNS.items():
    path = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{run_id}"
    try:
        run = api.run(path)
        rows = run.history(keys=list(METRICS.keys()), x_axis="_step", pandas=False)
        if not rows:
            print(f"  [{name}] no data returned")
            continue
        hist = {"step": np.array([r.get("_step", i) for i, r in enumerate(rows)], dtype=float)}
        for k in METRICS:
            hist[k] = np.array([r.get(k, float("nan")) for r in rows], dtype=float)
        run_histories[name] = hist
        print(f"  [{name}] fetched {len(hist['step'])} rows")
    except Exception as exc:
        print(f"  [{name}] WARNING: could not fetch run {run_id}: {exc}")

# ── Plot one figure per metric ────────────────────────────────────────────────
colors = {name: c for name, c in zip(LEARNING_CURVE_RUNS, px.colors.qualitative.Plotly)}

for metric_key, metric_label in METRICS.items():
    fig = go.Figure()

    for name, hist in run_histories.items():
        if metric_key not in hist:
            continue
        color = colors.get(name, "#888888")

        x = hist["step"]
        y = hist[metric_key]

        # Drop NaNs.
        mask = ~np.isnan(y)
        x, y = x[mask], y[mask]
        if len(y) == 0:
            continue

        # Apply rolling average for smoothing.
        y_smooth = rolling_mean(y, SMOOTH_WINDOW) if SMOOTH_WINDOW > 1 and len(y) >= SMOOTH_WINDOW else y

        # Raw (faint) trace.
        fig.add_trace(go.Scatter(
            x=x, y=y,
            mode="lines",
            name=name,
            line=dict(color=color, width=1),
            opacity=0.25,
            legendgroup=name,
            showlegend=False,
            hoverinfo="skip",
        ))
        # Smoothed trace.
        fig.add_trace(go.Scatter(
            x=x, y=y_smooth,
            mode="lines",
            name=name,
            line=dict(color=color, width=2),
            legendgroup=name,
            hovertemplate=f"{name}<br>step=%{{x}}<br>{metric_label}=%{{y:.4f}}<extra></extra>",
        ))

    fig.update_layout(
        title=f"Learning Curve — {metric_label} (smooth window={SMOOTH_WINDOW})",
        xaxis_title="Training Step",
        yaxis_title=metric_label,
        height=600,
        width=800,
        hovermode="x unified",
        autosize=True,
        margin=dict(l=60, r=20, t=60, b=120),
        legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
    )
    fig.show()


  [MAGEC] fetched 101 rows
  [Predictions] fetched 101 rows
  [Predictions+UQ (T=1.0)] fetched 101 rows
  [Predictions+UQ (T=0.5)] fetched 101 rows
  [Predictions+UQ (T=0.1)] fetched 101 rows
  [Observations (Boolean)] fetched 101 rows
  [Predictions+UQ (Boolean)] fetched 101 rows
